# Performs DPO on a SFTed model

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from matplotlib import pyplot as plt
import numpy as np

/Users/elly/anaconda3/envs/dpo/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# use if not installed already
# %pip install torch==2.6.0 #+cu124
# %pip install transformers==5.5.4 peft==0.19.1 matplotlib

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [5]:
# set up torch device
if torch.cuda.is_available():
    device = torch.device( "cuda" )
elif torch.backends.mps.is_available():
    device = torch.device( "mps" )
else:
    device = torch.device( "cpu" )
    
# load tokenizer and model
from pathlib import Path
import torch.nn as nn

model_name = "Qwen/Qwen2.5-0.5B_sft_step_13000"
tokenizer = AutoTokenizer.from_pretrained("checkpoints/" + model_name)
model = AutoModelForCausalLM.from_pretrained(
    "checkpoints/" + model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa"
)
print(f"Model Name: {model_name}")

class QwenRewardModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.model = base_model.model
        hidden_size = base_model.config.hidden_size
        self.reward_head = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, model_inputs):
        outputs = self.model(
            **model_inputs,
            output_hidden_states=False,
            return_dict=True,
            labels=None,
            use_cache=False,
        )
        hidden_states = outputs.last_hidden_state
        attention_mask = model_inputs.get("attention_mask", None)
        if attention_mask is not None:
            seq_lengths = attention_mask.sum(dim=1) - 1
            last_hidden = hidden_states[torch.arange(hidden_states.size(0)), seq_lengths]
        else:
            last_hidden = hidden_states[:, -1]
        reward = self.reward_head(last_hidden)
        return reward.squeeze(-1)

reward_model_name = "Qwen/Qwen2.5-0.5B_sft_step_13000_reward_step_7000"
reward_tokenizer = AutoTokenizer.from_pretrained("checkpoints/" + reward_model_name)
reward_base_model = AutoModelForCausalLM.from_pretrained(
    "checkpoints/" + reward_model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa"
)
reward_model = QwenRewardModel(reward_base_model)
reward_head_path = Path("checkpoints") / reward_model_name / "reward_head.pt"
reward_model.load_state_dict(torch.load(reward_head_path, map_location=next(reward_model.parameters()).device), strict=False)
reward_model = reward_model.to(torch.bfloat16).to(device)
reward_model.eval()
print(f"Reward Model Name: {reward_model_name}")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1016.05it/s]


Model Name: Qwen/Qwen2.5-0.5B_sft_step_13000


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 463.82it/s]


Reward Model Name: Qwen/Qwen2.5-0.5B_sft_step_13000_reward_step_7000


In [14]:
#test raw model
_prompt = "Describe the property of sin and cos functions. List the properties one by one."
_message = [{"role": "user", "content": _prompt}]
inputs = tokenizer.apply_chat_template(_message, add_generation_prompt=True, return_tensors="pt").to(model.device)
model.eval()
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True, 
        temperature=0.7,
        top_p=0.9,
    )
generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(f"Raw Model Generation:\n{generated_text}\n")


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Raw Model Generation:
system
You are a helpful assistant.
user
Describe the property of sin and cos functions. List the properties one by one.
assistant
Hello! I'd be happy to help you with your question. Here are the properties of the sine and cosine functions:

1. Periodicity: The sine and cosine functions are periodic functions with a period of 2π. This means that they repeat their values every 2π units.
2. Symmetry: The sine and cosine functions are odd functions, which means that they are symmetric with respect to the origin. That is, if you input an odd function into the sine function, you get the negative of the original function, and vice versa.
3. Domain and Range: The sine and cosine functions have a range of [-1, 1]. This means that their values can be any value between -1 and 1, inclusive.
4. Graphical Representation: The sine and cosine functions are graphs that oscillate between -1 and 1. The graph of the sine function is a wave that starts at the origin and gets closer a

In [ ]:
# pass model output through reward model
reward_inputs = reward_tokenizer(
    generated_text,
    return_tensors="pt",
).to(next(reward_model.parameters()).device)
reward_model.eval()
with torch.no_grad():
    reward = reward_model(reward_inputs)
print(reward)

tensor([-0.9883], device='mps:0', dtype=torch.bfloat16)


In [26]:
# load dataset 
# the train and test splits contain everything for sft and dpo, so we can use them for both stages
train_ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs")
test_ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="test_prefs")
# in sft stage, use 'message'
# in dpo stage, use 'chosen' and 'rejected'

print("Example Data:")
print("Train Data:")
print(train_ds[0])
print("Test Data:")
print(test_ds[0])

Example Data:
Train Data:
{'prompt': 'how can i develop a habit of drawing daily', 'prompt_id': '086b3e24f29b8956a01059f79c56db35d118a06fb6b844b095737d042795cd43', 'chosen': [{'content': 'how can i develop a habit of drawing daily', 'role': 'user'}, {'content': "Developing a daily habit of drawing can be challenging but with consistent practice and a few tips, it can become an enjoyable and rewarding part of your daily routine. Here are some strategies to help you develop the habit of drawing daily:\n\n1. Set a specific time: Allocate a specific time of the day to draw. It could be in the morning, afternoon, or evening. Make drawing a part of your daily routine.\n2. Set a specific duration: Determine the amount of time you want to spend on drawing each day. It can be as little as 10 minutes or as long as an hour. Be consistent with the duration to help build the habit.\n3. Start small and simple: Don't try to create a masterpiece every day, start with simple and easy-to-do sketches. Fo

Plan: use full finetuning for SFT, then use LoRA for DPO, because the model size is 0.5B.
At 0.5B, SFT may require a large representation shift. LoRA may be limiting.

In [27]:
train_ds[0]["messages"]
train_ds[0]["chosen"]
train_ds[0]["rejected"]

[{'content': 'how can i develop a habit of drawing daily', 'role': 'user'},
 {'content': "As an AI language model, I cannot personally develop habits for you. But, here are some tips for developing a habit of drawing daily:\n\n1. Start small: Start with simple drawings or doodles and gradually increase the complexity of your drawings.\n\n2. Set a schedule: Set a specific time every day to draw, and try to stick to it as much as possible.\n\n3. Make it fun: Don't pressure yourself to create masterpieces every time you draw. Make it a relaxing and enjoyable experience.\n\n4. Use resources: There are many drawing tutorials available online. Use resources like YouTube or online drawing courses to help you improve your skills.\n\n5. Surround yourself with inspiration: Expose yourself to a variety of art forms, such as paintings, illustrations, and photographs, to inspire and motivate you.\n\nRemember, everyone has their own creative style and pace. Just keep practicing and enjoying the proc

In [33]:
# set up hyperparameters
EPOCHS = 2
LEARNING_RATE = 1e-5
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
MAX_LENGTH = 1024
BETA = 0.1
LORA_RANK = 8
EXAMPLE_GEN_INTERVAL = 3000 # generate example outputs every EXAMPLE_GEN_INTERVAL optimization steps
EVAL_INTERVAL = 2  # validate every 1000 steps

print(f"Hyperparameters:\n \
        Epochs: {EPOCHS}\n \
        Learning Rate: {LEARNING_RATE}\n \
        Batch Size: {BATCH_SIZE}\n \
        Gradient Accumulation Steps: {GRADIENT_ACCUMULATION_STEPS}\n \
        Max Sequence Length: {MAX_LENGTH}\n \
        Beta (DPO): {BETA}\n \
        LoRA Rank: {LORA_RANK}\n \
        Example Generation Frequency (steps): {EXAMPLE_GEN_INTERVAL}\n \
        Validation Frequency (steps): {EVAL_INTERVAL}\n \n")

Hyperparameters:
         Epochs: 2
         Learning Rate: 1e-05
         Batch Size: 2
         Gradient Accumulation Steps: 8
         Max Sequence Length: 1024
         Beta (DPO): 0.1
         LoRA Rank: 8
         Example Generation Frequency (steps): 3000
         Validation Frequency (steps): 2
 



In [34]:
train_dataloader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [35]:
# prepare the model for lora
lora_config = LoraConfig(
    r=LORA_RANK, # the rank
    lora_alpha=16, # the scaling factor
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 4,202,496 || all params: 498,235,264 || trainable%: 0.8435


/Users/elly/anaconda3/envs/dpo/lib/python3.10/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/Users/elly/anaconda3/envs/dpo/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [7]:
# check out the injected lora layers!
trainable = [(n, p.shape) for n, p in model.named_parameters() if p.requires_grad]
print(f"Number of trainable parameters: {sum(p.numel() for n, p in model.named_parameters() if p.requires_grad)}")
print(f"Number of trainable layers: {len(trainable)}")
print("Trainable layers:")
for item in trainable:
    print(item)

Number of trainable parameters: 4202496
Number of trainable layers: 288
Trainable layers:
('base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', torch.Size([8, 896]))
('base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', torch.Size([896, 8]))
('base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', torch.Size([8, 896]))
('base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', torch.Size([128, 8]))
('base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', torch.Size([8, 896]))
('base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', torch.Size([896, 8]))
('base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', torch.Size([8, 896]))
('base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', torch.Size([4864, 8]))
('base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', torch.Size([8, 896]))
('base_model.model.model.layers.0.mlp.up_proj.l

In [36]:
# get reference model by no_grad and temporarily disabling LoRA
prompt = "Describe the property of sin and cos functions. List the properties one by one."
text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt").to(model.device)
model_inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

model.eval()
policy_output = model(**model_inputs)

model.eval()
with torch.no_grad():
    with model.disable_adapter():
        ref_outputs = model(**model_inputs)

print("policy logits shape:", policy_output.logits.shape)
print("ref logits shape:", ref_outputs.logits.shape)
print(policy_output.logits)

policy logits shape: torch.Size([1, 16, 151936])
ref logits shape: torch.Size([1, 16, 151936])
tensor([[[ 7.2188,  7.4375,  5.0000,  ..., -2.2031, -2.2031, -2.2031],
         [ 5.9375,  6.3750,  4.5000,  ..., -3.7500, -3.7500, -3.7500],
         [ 9.8125, 10.1250,  9.1875,  ..., -1.9609, -1.9609, -1.9609],
         ...,
         [11.0625, 10.0000,  9.4375,  ..., -3.2500, -3.2500, -3.2500],
         [13.6250,  9.0625,  9.8125,  ..., -2.2188, -2.2188, -2.2188],
         [ 7.0000, -3.6250,  4.8750,  ..., -5.3125, -5.3125, -5.3125]]],
       device='mps:0', dtype=torch.bfloat16, grad_fn=<LinearBackward0>)


In [ ]:
# model.save_pretrained(f"checkpoints/{model_name}_sft_epoch_0")
# model = AutoModelForCausalLM.from_pretrained(f"checkpoints/{model_name}_sft_epoch_0", device_map="auto", dtype=torch.bfloat16, attn_implementation="sdpa")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1232.85it/s]


In [37]:
# define train helpers

def unbatch_chat_messages(batched_messages):
    batch_size = len(batched_messages[0]["content"])
    return [
        [
            {
                "role": message_group["role"][sample_index],
                "content": message_group["content"][sample_index],
            }
            for message_group in batched_messages
        ]
        for sample_index in range(batch_size)
    ]

def mask_non_assistant_response(model_inputs, tokenizer):
    # the next position after <|im_start|>assistant
    # <|im_start|>assistant token id is 151644 followed by 77091
    start_pos = (model_inputs["input_ids"] == 151644) & (model_inputs["input_ids"].roll(-1, dims=1) == 77091)
    start_pos = start_pos * torch.arange(model_inputs["input_ids"].shape[1], device=model_inputs["input_ids"].device)
    start_pos = start_pos.max(dim=1).values + 2  # add 2 to get to the position of the first token of the response  
    # compute labels, only compute loss on assistant response
    labels = model_inputs["input_ids"].clone()
    pos = torch.arange(model_inputs["input_ids"].shape[1], device=model_inputs["input_ids"].device).unsqueeze(0) # (1, seq_len)
    mask = pos < start_pos.unsqueeze(1)  # (batch_size, seq_len)
    labels[mask] = tokenizer.pad_token_id  # set non-response tokens to padding
    return labels

def prepare_inputs(messages, tokenizer, device):
    conversations = unbatch_chat_messages(messages)
    texts = [
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        for messages in conversations
    ]
    model_inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(device)
    return model_inputs


In [38]:
# setup full-finetuning training loop for SFT
from tqdm import tqdm
import torch.optim as optim

# set up optimizer
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS*len(train_dataloader), eta_min=1e-6)

train_loss_history = [] # for each step (batch)
eval_loss_history = [] # for each validation step
eval_loss_steps = [] # step numbers when validation was performed
epoch_end_steps = [] # to keep track of step number at the end of each epoch for plotting
kl_div_history = [] # compare the output distribution of the current model and the reference model at each validation step
win_rate_history = [] # track the win rate of the current model against the reference model at each validation step
best_eval_loss = float('inf')
global_step = 0

def plot_loss_curves():
    # prepare arrays
    train_loss_history_arr = np.array(train_loss_history)
    eval_loss_history_arr = np.array(eval_loss_history)
    eval_loss_steps_arr = np.array(eval_loss_steps)
    step_num_history_arr = np.arange(1, len(train_loss_history) + 1)
    kl_div_history_arr = np.array(kl_div_history)
    win_rate_history_arr = np.array(win_rate_history)
    
    # create two subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10))
    
    # subplot 1: training and validation losses
    ax1.plot(step_num_history_arr, train_loss_history_arr, label='Training Loss', alpha=0.7)
    ax1.plot(eval_loss_steps_arr, eval_loss_history_arr, label='Validation Loss', marker='o', markersize=4, alpha=0.7)
    ax1.axhline(y=best_eval_loss, color='black', linestyle='--', linewidth=1, alpha=0.2, label=f'Min Eval Loss: {best_eval_loss:.4g}')
    
    # add vertical lines at epoch boundaries for both subplots
    for step_idx in epoch_end_steps:
        ax1.axvline(step_idx, color='gray', ls='--', lw=0.6, alpha=0.35)
        ax2.axvline(step_idx, color='gray', ls='--', lw=0.6, alpha=0.35)
    
    # add epoch ticks on top axis for subplot 1
    if len(epoch_end_steps) > 0:
        epoch_end_steps_arr = np.array(epoch_end_steps)
        epoch_ids = np.arange(1, len(epoch_end_steps_arr) + 1)
        max_labels = 12
        stride = max(1, int(np.ceil(len(epoch_end_steps_arr) / max_labels)))
        top_ticks = epoch_end_steps_arr[::stride]
        top_labels = epoch_ids[::stride]
        top_ax1 = ax1.secondary_xaxis('top')
        top_ax1.set_xticks(top_ticks)
        top_ax1.set_xticklabels(top_labels)
        top_ax1.set_xlabel('Epoch')
    
    ax1.set_xlabel('Step Number')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'Training & Validation Loss - {model_name.split("/")[-1]} DPO')
    ax1.legend(loc='upper right')
    ax1.grid(False)
    
    # subplot 2: win rate and KL divergence
    if len(win_rate_history_arr) > 0:
        ax2.plot(eval_loss_steps_arr, win_rate_history_arr, label='Win Rate', color='green', marker='o', markersize=4, alpha=0.7)
    if len(kl_div_history_arr) > 0:
        ax2.plot(eval_loss_steps_arr, kl_div_history_arr, label='KL Divergence', color='purple', marker='s', markersize=4, alpha=0.7)
    
    # add epoch ticks on top axis for subplot 2
    if len(epoch_end_steps) > 0:
        top_ax2 = ax2.secondary_xaxis('top')
        top_ax2.set_xticks(top_ticks)
        top_ax2.set_xticklabels(top_labels)
        top_ax2.set_xlabel('Epoch')
    
    ax2.set_xlabel('Step Number')
    ax2.set_ylabel('Value')
    ax2.set_title(f'Win Rate & KL Divergence - {model_name.split("/")[-1]} DPO')
    ax2.legend(loc='upper right')
    ax2.grid(False)
    
    fig.tight_layout()
    fig.savefig(f"dpo_loss_curve.png")
    
for epoch in range(EPOCHS):
    # train
    pbar = tqdm(train_dataloader, desc=f"Train Epoch {epoch + 1}/{EPOCHS}")
    optimizer.zero_grad()
    accumulation_step = 0
    for batch in pbar:
        model.train()
        win_model_inputs = prepare_inputs(batch["chosen"], tokenizer, model.device)
        win_labels = mask_non_assistant_response(win_model_inputs, tokenizer)
        lose_model_inputs = prepare_inputs(batch["rejected"], tokenizer, model.device)
        lose_labels = mask_non_assistant_response(lose_model_inputs, tokenizer)
        win_input_ids_shifted = torch.roll(win_model_inputs["input_ids"], shifts=-1, dims=1)
        lose_input_ids_shifted = torch.roll(lose_model_inputs["input_ids"], shifts=-1, dims=1)
        
        # forward passes
        lora_win_outputs = model(**win_model_inputs, labels=None, use_cache=False) # disable KV cache
        lora_lose_outputs = model(**lose_model_inputs, labels=None, use_cache=False) # disable KV cache
        lora_win_log_prob = torch.nn.functional.log_softmax(lora_win_outputs.logits, dim=-1)
        lora_lose_log_prob = torch.nn.functional.log_softmax(lora_lose_outputs.logits, dim=-1)
        # consider log probs of only token positions in training data
        lora_win_log_prob_gathered = torch.gather(lora_win_log_prob, dim=2, index=win_input_ids_shifted.unsqueeze(-1)).squeeze(-1)
        lora_lose_log_prob_gathered = torch.gather(lora_lose_log_prob, dim=2, index=lose_input_ids_shifted.unsqueeze(-1)).squeeze(-1)
        win_mask = torch.roll(win_labels != tokenizer.pad_token_id, shifts=-1, dims=1) # shift the mask to align with the log probs of the next tokens
        lose_mask = torch.roll(lose_labels != tokenizer.pad_token_id, shifts=-1, dims=1)
        lora_win_sentence_log_prob = (lora_win_log_prob_gathered * win_mask)[:, :-1].sum(dim=1)
        lora_lose_sentence_log_prob = (lora_lose_log_prob_gathered * lose_mask)[:, :-1].sum(dim=1)

        # get reference model outputs without gradient and adapter
        with torch.no_grad():
            with model.disable_adapter():
                model.eval() # make sure to set reference model to eval mode for stable evaluation!
                ref_win_outputs = model(**win_model_inputs, labels=None, use_cache=False)
                ref_lose_outputs = model(**lose_model_inputs, labels=None, use_cache=False)
                ref_win_log_prob = torch.nn.functional.log_softmax(ref_win_outputs.logits, dim=-1)
                ref_lose_log_prob = torch.nn.functional.log_softmax(ref_lose_outputs.logits, dim=-1)
                ref_win_log_prob_gathered = torch.gather(ref_win_log_prob, dim=2, index=win_input_ids_shifted.unsqueeze(-1)).squeeze(-1)
                ref_lose_log_prob_gathered = torch.gather(ref_lose_log_prob, dim=2, index=lose_input_ids_shifted.unsqueeze(-1)).squeeze(-1)
                # win_mask and lose_mask are the same for LoRA and reference since they are based on the input labels
                ref_win_sentence_log_prob = (ref_win_log_prob_gathered * win_mask)[:, :-1].sum(dim=1)
                ref_lose_sentence_log_prob = (ref_lose_log_prob_gathered * lose_mask)[:, :-1].sum(dim=1)
        model.train() # switch back to train mode for the policy model after getting reference outputs
        # compute DPO loss
        loss = -torch.log(
            torch.sigmoid(
                BETA * (
                    lora_win_sentence_log_prob - lora_lose_sentence_log_prob
                    - ref_win_sentence_log_prob + ref_lose_sentence_log_prob
                )
            )
        )
        # here, loss has shape [B], each entry is the loss of i-th batch!
        # remember batch reduction! take mean of the loss.
        loss = torch.mean(loss)
        (loss / GRADIENT_ACCUMULATION_STEPS).backward()
        accumulation_step += 1
        if accumulation_step % GRADIENT_ACCUMULATION_STEPS == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        train_loss_history.append(loss.detach().item())
        global_step += 1
        
        # Step-based validation
        if global_step % EVAL_INTERVAL == 0:
            model.eval()
            pbar_eval = tqdm(test_dataloader, desc=f"Eval at Step {global_step}", leave=False)
            eval_loss = 0
            eval_batch_count = 0
            kl_div = 0
            for batch_eval in pbar_eval:
                # compute DPO loss on the eval set
                win_model_inputs_eval = prepare_inputs(batch_eval["chosen"], tokenizer, model.device)
                win_labels_eval = mask_non_assistant_response(win_model_inputs_eval, tokenizer)
                lose_model_inputs_eval = prepare_inputs(batch_eval["rejected"], tokenizer, model.device)
                lose_labels_eval = mask_non_assistant_response(lose_model_inputs_eval, tokenizer)
                win_input_ids_shifted_eval = torch.roll(win_model_inputs_eval["input_ids"], shifts=-1, dims=1)
                lose_input_ids_shifted_eval = torch.roll(lose_model_inputs_eval["input_ids"], shifts=-1, dims=1)
                # forward passes
                with torch.no_grad():
                    lora_win_outputs_eval = model(**win_model_inputs_eval, labels=None, use_cache=False) # disable KV cache
                    lora_lose_outputs_eval = model(**lose_model_inputs_eval, labels=None, use_cache=False) # disable KV cache
                    ref_win_outputs_eval = model(**win_model_inputs_eval, labels=None, use_cache=False)
                    ref_lose_outputs_eval = model(**lose_model_inputs_eval, labels=None, use_cache=False)
                    lora_win_log_prob_eval = torch.nn.functional.log_softmax(lora_win_outputs_eval.logits, dim=-1)
                    lora_lose_log_prob_eval = torch.nn.functional.log_softmax(lora_lose_outputs_eval.logits, dim=-1)
                    ref_win_log_prob_eval = torch.nn.functional.log_softmax(ref_win_outputs_eval.logits, dim=-1)
                    ref_lose_log_prob_eval = torch.nn.functional.log_softmax(ref_lose_outputs_eval.logits, dim=-1)
                    lora_win_log_prob_gathered_eval = torch.gather(lora_win_log_prob_eval, dim=2, index=win_input_ids_shifted_eval.unsqueeze(-1)).squeeze(-1)
                    lora_lose_log_prob_gathered_eval = torch.gather(lora_lose_log_prob_eval, dim=2, index=lose_input_ids_shifted_eval.unsqueeze(-1)).squeeze(-1)
                    ref_win_log_prob_gathered_eval = torch.gather(ref_win_log_prob_eval, dim=2, index=win_input_ids_shifted_eval.unsqueeze(-1)).squeeze(-1)
                    ref_lose_log_prob_gathered_eval = torch.gather(ref_lose_log_prob_eval, dim=2, index=lose_input_ids_shifted_eval.unsqueeze(-1)).squeeze(-1)
                    win_mask_eval = torch.roll(win_labels_eval != tokenizer.pad_token_id, shifts=-1, dims=1) # shift the mask to align with the log probs of the next tokens
                    lose_mask_eval = torch.roll(lose_labels_eval != tokenizer.pad_token_id, shifts=-1, dims=1)
                    lora_win_sentence_log_prob_eval = (lora_win_log_prob_gathered_eval * win_mask_eval)[:, :-1].sum(dim=1)
                    lora_lose_sentence_log_prob_eval = (lora_lose_log_prob_gathered_eval * lose_mask_eval)[:, :-1].sum(dim=1)
                    ref_win_sentence_log_prob_eval = (ref_win_log_prob_gathered_eval * win_mask_eval)[:, :-1].sum(dim=1)
                    ref_lose_sentence_log_prob_eval = (ref_lose_log_prob_gathered_eval * lose_mask_eval)[:, :-1].sum(dim=1)
                    
                    loss = -torch.log(
                        torch.sigmoid(
                            BETA * (
                                lora_win_sentence_log_prob_eval - lora_lose_sentence_log_prob_eval
                                - ref_win_sentence_log_prob_eval + ref_lose_sentence_log_prob_eval
                            )
                        )
                    )
                    loss = torch.mean(loss)
                    eval_loss += loss.detach().item()
                    eval_batch_count += 1
                    
                    # compute KL divergence between current model and reference model
                    kl_div += torch.nn.functional.kl_div(
                        lora_win_log_prob_eval, ref_win_log_prob_eval, reduction='batchmean', log_target=True
                    ) + torch.nn.functional.kl_div(
                        lora_lose_log_prob_eval, ref_lose_log_prob_eval, reduction='batchmean', log_target=True
                    )
            kl_div /= eval_batch_count
            kl_div_history.append(kl_div.detach().item() / 2.0)
            # compute win rate: fraction of samples where chosen has higher log prob than rejected
            eval_win_rate = torch.mean((lora_win_sentence_log_prob_eval > lora_lose_sentence_log_prob_eval).float()).item()
            win_rate_history.append(eval_win_rate)
            eval_loss /= eval_batch_count
            eval_loss_history.append(eval_loss)
            eval_loss_steps.append(global_step)
            tqdm.write(f"Step {global_step}: Train Loss={train_loss_history[-1]:.4f}, Eval Loss={eval_loss:.4f}, Win Rate={eval_win_rate:.4f}")
            
            # update best eval loss
            if eval_loss < best_eval_loss:
                best_eval_loss = eval_loss
                model.save_pretrained(f"checkpoints/{model_name}_dpo_step_{global_step}")
                tokenizer.save_pretrained(f"checkpoints/{model_name}_dpo_step_{global_step}")
                print(f"New best model saved with eval loss {best_eval_loss:.4f}")
            
            plot_loss_curves()
        
        # # in each validation, generate for several prompts in test set and evaluate the reward win rate against reference model
        # if global_step % EVAL_INTERVAL == 0:
        #     model.eval()
        #     pbar_eval = tqdm(test_dataloader, desc=f"Eval at Step {global_step}", leave=False)
        
        if global_step % EXAMPLE_GEN_INTERVAL == 0:
            # example generation
            model.eval()
            _prompt = "Describe the property of sin and cos functions. List the properties one by one."
            _message = [{"role": "user", "content": _prompt}]
            inputs = tokenizer.apply_chat_template(_message, add_generation_prompt=True, return_tensors="pt").to(model.device)
            with torch.no_grad():
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=512,
                    do_sample=True, 
                    temperature=0.7,
                    top_p=0.9,
                )
            generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            print(f"Example Generation at Epoch {epoch + 1} Step {accumulation_step + 1}:\n{generated_text}\n")

    if accumulation_step % GRADIENT_ACCUMULATION_STEPS != 0:
        model.train()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    epoch_end_steps.append(len(train_loss_history))
    



Train Epoch 1/2:   0%|          | 1/30568 [00:21<182:48:37, 21.53s/it]


KeyboardInterrupt: 

In [ ]:
# # for quick debug: shape check
# prompt = "Describe the property of sin and cos functions. List the properties one by one."
# text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt").to(model.device)
# model_inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# shifted_input_ids = torch.roll(model_inputs["input_ids"], shifts=-1, dims=1)
# print(f"Shifted input IDs: {shifted_input_ids}") # shift position for next-token prediction

# # forward passes
# output_shape = torch.Size([1, 16, 151936])
# print("Output shape:", output_shape)
# outputs = torch.rand(output_shape).to(model.device)
# softmax_outputs = torch.nn.functional.softmax(outputs, dim=-1)
# print("Softmax output", softmax_outputs)
# log_prob = torch.log(softmax_outputs + 1e-8)
# gathered_log_prob = torch.gather(log_prob, dim=2, index=shifted_input_ids.unsqueeze(-1)).squeeze(-1)[:, :-1]

# sentence_log_prob = gathered_log_prob.sum(dim=1)
# print("Gathered log prob", gathered_log_prob)
# print("Sentence log prob", sentence_log_prob)


Shifted input IDs: tensor([[  279,  3343,   315,  7437,   323,  7960,  5746,    13,  1759,   279,
          5888,   825,   553,   825,    13, 74785]], device='mps:0')
Output shape: torch.Size([1, 16, 151936])
Softmax output tensor([[[7.7372e-06, 5.2464e-06, 8.3718e-06,  ..., 8.7940e-06,
          5.0799e-06, 1.0391e-05],
         [5.9999e-06, 4.0122e-06, 8.8373e-06,  ..., 5.2177e-06,
          5.7388e-06, 5.5652e-06],
         [4.9094e-06, 7.4732e-06, 5.7025e-06,  ..., 4.3935e-06,
          1.0039e-05, 6.5480e-06],
         ...,
         [9.4026e-06, 5.7481e-06, 8.5969e-06,  ..., 4.6160e-06,
          3.9365e-06, 6.8088e-06],
         [4.5374e-06, 9.9039e-06, 4.9073e-06,  ..., 4.0946e-06,
          7.0532e-06, 9.0639e-06],
         [7.4167e-06, 9.1232e-06, 5.7632e-06,  ..., 6.7512e-06,
          5.2117e-06, 4.0058e-06]]], device='mps:0')
Gathered log prob tensor([[-11.6033, -12.3475, -11.9688, -11.8710, -11.6384, -11.8008, -11.5342,
         -11.7771, -12.1163, -11.7601, -12.2187, -11.

In [ ]:
# import torch

# a = torch.tensor([[1, 2, 3]], dtype=torch.long)        # indices
# b = torch.tensor([[[0,1,2,3],
#                    [0,1,2,3],
#                    [0,1,2,3]]])                         # values

# # gather along last dim (dim=2); index must have same leading dims and an extra dim at gather axis
# res = torch.gather(b, dim=2, index=a.unsqueeze(-1)).squeeze(-1)
# print(res)  # tensor([[1,2,3]])

tensor([[1, 2, 3]])
